In [516]:
import random
from fractions import Fraction
from typing import List, Tuple
import pandas as pd
from datasets import Dataset

import pandas as pd 
import os 

from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

import seaborn as sns

from datasets import load_dataset, load_from_disk
import numpy as np
import json

from utils.corrupt_num import *
from utils.chunk_r import *

## Evals - How to Scale:
1. number of operands
2. new operations

In [169]:
nums = [ 47, 2, 42, 9, 62, 20, 64 ]
target = 228
question = f"Using the numbers {nums}, create an equation that equals {target}. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once"

In [170]:
tokenizer = AutoTokenizer.from_pretrained("zwhe99/DeepMath-1.5B")

In [171]:
conversations = [
    {"role": "system", "content": "Please reason and answer the following question. You should return the final answer in <answer></answer> tags, for example <answer>(1 + 2) / 3</answer>."},
    {"role": "user", "content": question}
]
prompt = tokenizer.apply_chat_template(
    conversations,
    tokenize=False,
    add_generation_prompt=True
)

In [172]:
print(prompt)

<｜begin▁of▁sentence｜>Please reason and answer the following question. You should return the final answer in <answer></answer> tags, for example <answer>(1 + 2) / 3</answer>.<｜User｜>Using the numbers [47, 2, 42, 9, 62, 20, 64], create an equation that equals 228. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once<｜Assistant｜><think>



In [ ]:
"Please reason and answer the following question. At the end, you should return the final answer in <answer></answer> tags, for example <answer>(1 + 2) / 3</answer>. Using the numbers {nums}, create an equation that equals {target}. You can use basic arithmetic operations (+, -, *, /) in the expression and each number can only be used once."

In [173]:
47 + 62 + 9 + 64 + 42/2

203.0

# Check Eval Dataset

In [806]:
cd /home/al2644/research/codebase/reasoning/rlvr/

/home/al2644/research/codebase/reasoning/rlvr


In [808]:
path = "./data/train/countdown/train.parquet"
original_dataset = pd.read_parquet(path)

path = "./data/train/countdown_level45_20k/train.parquet"
new_dataset = pd.read_parquet(path)

In [382]:
def format_dataset(row, tokenizer = tokenizer):
    problem = row['problem']
    response = row['response']
    conversation = [
        {"role": "user", "content": problem},
        {"role": "assistant", "content": response}
    ]

    return tokenizer.apply_chat_template(conversation, tokenize=False)

In [383]:
result = format_dataset(row)

# Reasoning Benchmark Eval

In [614]:
root = "./results/countdown/benchmark/"
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-3B')
for fname in os.listdir(root):
    if 'correct' not in fname:
        print(fname)
        df = pd.read_pickle(os.path.join(root, fname))
        print(f"Accuracy: {df['correct'].mean()}")

# model specific
df = pd.read_pickle(os.path.join(root, 'Qwen2.5-3B-grpo-countdown-level4-5-1epoch.pickle'))

Qwen2.5-3B-distill-countdown-level3-4-1epoch.pickle
Accuracy: 0.24214285714285713
Qwen2.5-3B-distill-countdown-level3-4-3epoch.pickle
Accuracy: 0.24714285714285714
Qwen2.5-3B-grpo-countdown-level4-5-1epoch.pickle
Accuracy: 0.35


In [612]:
def groupby_level (df):
    stats = df.groupby('level')['correct'].mean()
    return stats

# Corrupted Numbers

In [795]:
root = "./results/countdown/corrupt_numbers/"

def has_answer (response):
    if "<answer>" in response:
        return 1.
    else:
        return 0.
    
for fname in os.listdir(root):
    if 'still_correct' not in fname:
        print(fname)
        df = pd.read_pickle(os.path.join(root, fname))
        print(f"Accuracy: {df['still_correct'].mean()}")

Qwen2.5-3B-distill-countdown-level3-4-1epoch.pickle
Accuracy: 0.5202702702702703
Qwen2.5-3B-grpo-countdown-level4-5-1epoch.pickle
Accuracy: 0.5934684684684685


In [694]:
wrong_answer = df[(df['still_correct']==0.) & (df['if_answer'])]
correct_answer = df[df['still_correct']==1.]

In [695]:
wrong_answer[['level', 'type']].value_counts()

level  type         
5      post_</think>    80
6      post_</think>    21
5      pre_</think>     17
6      pre_</think>      5
Name: count, dtype: int64

In [705]:
print(df[df['type'] == 'pre_<think>']['prompt'].iloc[10])

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Please reason and answer the following question. Using the numbers [35, 48, 94, 46, 4], create an equation that equals 140. You can only use basic arithmetic operations (+, -, *, /) in the expression and each number should be used exactly once. You should report the answer equation expression in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.

This is user's thought process for this problem: <think>
Okay, let's see. I need to use the numbers 35, 48, 94, 46, and 4 exactly once each, with basic arithmetic operations to make 140. Hmm. Let me start by thinking about possible combinations. 

First, maybe I should check if adding some numbers and subtracting others could work. Let's list the numbers again: 35, 48, 94, 46, 4. The target is 140. Let me try some combinations.

What if I subtract two numbers? For example, 94 minus something? Let's see. If I take 94 and subtract 46, that gives 48. Then

In [706]:
idx = 21
example = correct_answer.iloc[idx]
prompt, response = example['prompt'], example['post_corruption_response']

In [707]:
print(prompt)

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Please reason and answer the following question. Using the numbers [43, 77, 79, 50, 96], create an equation that equals 259. You can only use basic arithmetic operations (+, -, *, /) in the expression and each number should be used exactly once. You should report the answer equation expression in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.<|im_end|>
<|im_start|>assistant
<think>
Okay, let's see. I need to use the numbers 43, 77, 79, 50, and 96 exactly once each with basic operations to make 259. Hmm. Let me start by thinking about possible combinations.

First, maybe I should check if adding some numbers and subtracting others could work. Let me try adding two numbers and subtracting the others. For example, 96 + 79 is 175. Then 77 + 50 is 127. Then 175 + 127 is 302. That's too high. Maybe subtract something? If I subtract 43, 302 - 43 is 259. Wait, that
seems promising. Let me check:




In [708]:
print(response)

the original numbers? Wait, no, the original numbers are 43, 77, 79, 50, 96. So maybe that's not the right approach. Wait, maybe I need to use all numbers once in the equation. So the idea would be (96 + 79) + (77 + 50) - 43 = 259. Let me compute that:

96 + 79 = 175. 77 + 50 = 127. 175 + 127 = 302. 302 - 43 = 259. Yes! That works. But wait, how do I arrange the operations? Let me check that each number is used exactly once. 96, 79, 77, 50, 43. Yes, all are used once. So the equation would be (96 + 79) + (77 + 50) - 43 = 259. Let me verify again:

96 + 79 is 175. 77 + 50 is 127. 175 + 127 is 302. 302 - 43 is 259. Correct. So that's the answer. But maybe I need to check if there's another way, but this seems to work. Let me write it properly in the required format. The expression would be (96 + 79) + (77 + 50) - 43. So that's the answer.
</think>

<answer> (96 + 79) + (77 + 50) - 43 </answer>



# Distractor Injection

In [796]:
root = "./results/countdown/inject_distractor/"

def has_answer (response):
    if "<answer>" in response:
        return 1.
    else:
        return 0.
    
for fname in os.listdir(root):
    print(fname)
    df = pd.read_pickle(os.path.join(root, fname))
    print(f"Original Accuracy: {df['original_correct'].mean()}")
    print(f"Distractor Accuracy: {df['distractor_correct'].mean()}")

df = pd.read_pickle(os.path.join(root, "Qwen2.5-3B-grpo-countdown-level4-5-1epoch.pickle"))

Qwen2.5-3B-distill-countdown-level3-4-1epoch.pickle
Original Accuracy: 0.3626237623762376
Distractor Accuracy: 0.5006188118811881
Qwen2.5-3B-grpo-countdown-level4-5-1epoch.pickle
Original Accuracy: 0.46163366336633666
Distractor Accuracy: 0.47277227722772275


In [797]:
df = pd.read_pickle(os.path.join(root, "Qwen2.5-3B-distill-countdown-level3-4-1epoch.pickle"))
df.groupby("prefix_ratio")[["original_correct", "distractor_correct"]].mean()

,original_correct,distractor_correct
prefix_ratio,,
0.00,0.047030,0.816832
0.25,0.544554,0.292079
0.50,0.462871,0.400990
0.75,0.396040,0.492574


In [798]:
df = pd.read_pickle(os.path.join(root, "Qwen2.5-3B-grpo-countdown-level4-5-1epoch.pickle"))
df.groupby("prefix_ratio")[["original_correct", "distractor_correct"]].mean()

,original_correct,distractor_correct
prefix_ratio,,
0.00,0.009901,0.938119
0.25,0.670792,0.252475
0.50,0.613861,0.309406
0.75,0.551980,0.391089


In [799]:
df

,nums,target,example_solution,problem,level,task,original_response,reasoning_chunks,prefix_ratio,distractor_ratio,distract_nums,distract_target,distract_reasoning,prompt,post_distraction_response,original_correct,distractor_correct
0,"[50, 77, 64, 67, 73]",331,(((73+50)+64)+(67+77)),Please reason and answer the following questio...,5,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.0,0.25,"[86, 91, 87, 68, 77, 66]",165,"<think>\nOkay, let's see. I need to use the nu...",<|im_start|>system\nYou are a helpful assistan...,"Not sure. Alternatively, 87 - 66 = 21. 91 - 7...",0.0,1.0
1,"[50, 77, 64, 67, 73]",331,(((73+50)+64)+(67+77)),Please reason and answer the following questio...,5,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.0,0.5,"[20, 73, 85, 82, 33]",89,"<think>\nOkay, let's see. I need to use the nu...",<|im_start|>system\nYou are a helpful assistan...,33 + 85 is 118. 118 - 82 is 36. 36 +73 is 109...,0.0,1.0
2,"[50, 77, 64, 67, 73]",331,(((73+50)+64)+(67+77)),Please reason and answer the following questio...,5,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.0,0.75,"[20, 73, 85, 82, 33]",89,"<think>\nOkay, let's see. I need to use the nu...",<|im_start|>system\nYou are a helpful assistan...,So the answer would be that expression. So th...,0.0,1.0
3,"[50, 77, 64, 67, 73]",331,(((73+50)+64)+(67+77)),Please reason and answer the following questio...,5,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.0,1.0,"[74, 51, 64, 31, 48]",104,"<think>\nOkay, let's see. I need to use the nu...",<|im_start|>system\nYou are a helpful assistan...,The answer should be in <answer> tags. So <an...,0.0,1.0
4,"[50, 77, 64, 67, 73]",331,(((73+50)+64)+(67+77)),Please reason and answer the following questio...,5,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.25,0.25,"[25, 87, 42, 34, 57]",63,"<think>\nOkay, let's see. I need to use each o...",<|im_start|>system\nYou are a helpful assistan...,"87 - 57 is 30, then 30 - 25 is 5, not helpful....",1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1611,"[84, 3, 85, 12, 51, 9]",307,(((9+85)+12)+((3*84)-51)),Please reason and answer the following questio...,6,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.5,1.0,"[75, 96, 96, 56, 68]",129,"<think>\nOkay, let's see. I need to use each o...",<|im_start|>system\nYou are a helpful assistan...,"Therefore, the equation is 96 - 75 + 96 + 68 ...",1.0,0.0
1612,"[84, 3, 85, 12, 51, 9]",307,(((9+85)+12)+((3*84)-51)),Please reason and answer the following questio...,6,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.75,0.25,"[74, 51, 64, 31, 48]",104,"<think>\nOkay, let's see. I need to use each o...",<|im_start|>system\nYou are a helpful assistan...,Maybe (64 - 51) = 13. Then 13 + 48 = 61. Then...,1.0,0.0
1613,"[84, 3, 85, 12, 51, 9]",307,(((9+85)+12)+((3*84)-51)),Please reason and answer the following questio...,6,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.75,0.5,"[71, 93, 76, 10, 96]",184,"<think>\nOkay, let's see. I need to use each o...",<|im_start|>system\nYou are a helpful assistan...,204 is not 184. What if I do 76 + 93 = 169. 16...,0.0,0.0
1614,"[84, 3, 85, 12, 51, 9]",307,(((9+85)+12)+((3*84)-51)),Please reason and answer the following questio...,6,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.75,0.75,"[17, 76, 59, 23, 18]",41,"<think>\nOkay, let's see. I need to use each o...",<|im_start|>system\nYou are a helpful assistan

In [802]:
idx = 7
example = df.iloc[idx]
prefix_ratio, distractor_ratio = example["prefix_ratio"], example["distractor_ratio"]
print("prefix_ratio: ", prefix_ratio)
print("distractor_ratio: ", distractor_ratio)
prompt, response, original_correct, distractor_correct = example["prompt"], example["post_distraction_response"], example["original_correct"], example["distractor_correct"]

print("original correct?: ", original_correct)
print("distract correct?: ", distractor_correct)

prefix_ratio:  0.25
distractor_ratio:  1.0
original correct?:  0.0
distract correct?:  1.0


In [803]:
print(prompt)

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Please reason and answer the following question. Using the numbers [50, 77, 64, 67, 73], create an equation that equals 331. You can only use basic arithmetic operations (+, -, *, /) in the expression and each number should be used exactly once. You should report the answer equation expression in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.<|im_end|>
<|im_start|>assistant
<think>
Okay, let's see. I need to use each of the numbers 50, 77, 64, 67, and 73 exactly once with basic operations to make 331. Hmm. Let me think about possible combinations.Wait. Let me think again: Okay, let's see. I need to use the numbers 86, 91, 87, 68, 77, and 66 exactly once each, with basic arithmetic operations to make 165. Hmm, first, maybe I should think about possible combinations. Let me list the numbers again: 86, 91, 87, 68, 77, 66. All have to be used once. The target is 165.What's 165? Let me think abo

In [804]:
print(response)

 Which is the same as the previous grouping. Alternatively, written as (66 + 77) + (86 - 68) + (91 - 87). Either way, the operations are all allowed. Let me check again if the order is correct. Let me compute the expression step by step. 66 + 77 is 143. Then 143 + 86 is 229. Then subtract 68: 229 - 68 = 161. Then add 91: 161 + 91 = 252. Then subtract 87: 252 - 87 = 165. Correct. So that's the right equation.
</think>

<answer> 66 + 77 + 86 - 68 + 91 - 87 </answer>

